In [1]:
!pip install wandb xgboost scikit-learn -q

In [2]:
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/dermatology/dermatology.data -qq

In [3]:
import os
import numpy as np
import xgboost as xgb
import wandb
from sklearn.metrics import classification_report, accuracy_score, f1_score

In [8]:
wandb.login()
run = wandb.init(project="Lab1-visualize-models", name="dermatology-xgb-tuned")

In [9]:
data = np.loadtxt(
    "./dermatology.data",
    delimiter=",",
    converters={33: lambda x: int(x == b"?"), 34: lambda x: int(x) - 1},
)

In [10]:
# shuffling
rng = np.random.default_rng(42)
rng.shuffle(data)

split_ratio = 0.75
split_idx = int(data.shape[0] * split_ratio)

train_X, train_Y = data[:split_idx, :33], data[:split_idx, 34]
test_X, test_Y = data[split_idx:, :33], data[split_idx:, 34]

print(f"Training samples: {train_X.shape[0]}  |  Test samples: {test_X.shape[0]}")
print(f"Feature count: {train_X.shape[1]}  |  Classes: {len(np.unique(train_Y))}")

# logging the dtaaset
wandb.log({
    "dataset/train_size": train_X.shape[0],
    "dataset/test_size": test_X.shape[0],
    "dataset/num_features": train_X.shape[1],
})

dtrain = xgb.DMatrix(train_X, label=train_Y)
dtest = xgb.DMatrix(test_X, label=test_Y)

params = {
    "objective": "multi:softprob",
    "eval_metric": "mlogloss",
    "eta": 0.15,
    "max_depth": 5,
    "subsample": 0.8,
    "colsample_bytree": 0.85,
    "num_class": 6,
    "verbosity": 0,
    "seed": 42,
}
wandb.config.update(params)

num_rounds = 25
evals = [(dtrain, "train"), (dtest, "val")]

model = xgb.train(
    params,
    dtrain,
    num_rounds,
    evals=evals,
    callbacks=[wandb.xgboost.WandbCallback()],
    verbose_eval=5,
)

prob_preds = model.predict(dtest)
pred_labels = np.argmax(prob_preds, axis=1)

accuracy = accuracy_score(test_Y, pred_labels)
f1_macro = f1_score(test_Y, pred_labels, average="macro")
error_rate = 1.0 - accuracy

print(f"\nAccuracy : {accuracy:.4f}")
print(f"F1 Macro : {f1_macro:.4f}")
print(f"Error Rate: {error_rate:.4f}")

print("\n********** Classification Report *******************")
class_names = [f"class_{i}" for i in range(6)]
print(classification_report(test_Y, pred_labels, target_names=class_names))

# logging metrices
run.summary["accuracy"] = accuracy
run.summary["f1_macro"] = f1_macro
run.summary["error_rate"] = error_rate

# confusion matrix via wandb
wandb.sklearn.plot_confusion_matrix(test_Y, pred_labels, labels=class_names)

importance = model.get_score(importance_type="gain")
if importance:
    feat_table = wandb.Table(
        columns=["feature", "gain"],
        data=sorted(importance.items(), key=lambda x: x[1], reverse=True),
    )
    wandb.log({"feature_importance": feat_table})

run.finish()
print("Run successfully completed")



Training samples: 274  |  Test samples: 92
Feature count: 33  |  Classes: 6
[0]	train-mlogloss:1.36570	val-mlogloss:1.38440
[5]	train-mlogloss:0.61556	val-mlogloss:0.67647
[10]	train-mlogloss:0.32258	val-mlogloss:0.38570
[15]	train-mlogloss:0.18189	val-mlogloss:0.25390
[20]	train-mlogloss:0.11074	val-mlogloss:0.18330
[24]	train-mlogloss:0.07902	val-mlogloss:0.15010

Accuracy : 0.9783
F1 Macro : 0.9781
Error Rate: 0.0217

********** Classification Report *******************
              precision    recall  f1-score   support

     class_0       0.97      1.00      0.98        30
     class_1       0.93      0.93      0.93        14
     class_2       1.00      1.00      1.00        21
     class_3       1.00      0.92      0.96        12
     class_4       1.00      1.00      1.00         9
     class_5       1.00      1.00      1.00         6

    accuracy                           0.98        92
   macro avg       0.98      0.97      0.98        92
weighted avg       0.98      0.98 

dataset/num_features,▁
dataset/test_size,▁
dataset/train_size,▁
epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
train-mlogloss,█▇▆▅▄▄▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
val-mlogloss,█▇▆▅▄▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
accuracy,0.97826
dataset/num_features,33
dataset/test_size,92
dataset/train_size,274
epoch,24


Run successfully completed
